In [1]:
import numpy as np
import numba as nb

In [2]:
from math import lgamma

@nb.njit
def B(d):
    '''-
    Returns the hyper-volume of the unit ball
    in d dimensions

    https://en.wikipedia.org/wiki/N-sphere#Volume_and_area
    '''
    gamma_of_1_plus_d_over_2 = np.exp(lgamma(1 + d/2))
    return np.pi**(d/2) / gamma_of_1_plus_d_over_2

In [3]:
@nb.njit
def u(r, E, sigma):
	'''
	pair potential for penetrable spheres or disks given a radius
	'''
	return E if r < sigma else (E/2 if r == sigma else 0)

In [ ]:
from random import shuffle



@nb.njit
def __repopulate_cell_list(
    bin_indices, cell_list, cell_list_size,
    particle_number, spatial_dimension_count,
    divisions):
    '''
    takes in the array of positions and fills them
    into the cell list
    '''
    # clearing all lists entries
    for i in range(cell_list_size):
        cell_list[i].clear()
	# bin indices are given by flooring
    
    for k in range(particle_number):
        # flattening the indices ...
        b_k = bin_indices[k]
		# by: b_k[0] + divisions * b_k[1] + ... + divisions**(d-1) * b_k[d-1]
        flattened_index = 0
        for l in range(spatial_dimension_count):
            flattened_index += int(b_k[l] * divisions**l)
        cell_list[flattened_index].append(k)



@nb.njit
def run(
    overlap_energy, kB_times_temperature,
    packing_density, particle_number, spatial_dimension_count=3,
    simulation_box_size=1, move_count=10000):
    '''
    Executing the metropolis hastings sampling of
    the penetrable sphere system
    '''
    # computing the particle diameter corresponding to the ...
    # ... sought packing density
    volume = simulation_box_size**spatial_dimension_count
    number_density = particle_number / volume
    sigma = 2 * (
        packing_density / (
            B(spatial_dimension_count) * number_density)
            )**(1/spatial_dimension_count)
    # randomly placed particles
    x = np.random.rand(particle_number, spatial_dimension_count)
    # cell list for energy computation
    # TODO ROUND DOWN TO CLOSEST NON-NEGATIVE INTEGER MULTIPLE OF 3
    divisions = int(np.floor(simulation_box_size / sigma)) 
    cell_list_size = divisions**spatial_dimension_count
    cell_list = nb.typed.List([
        # the number of divisions per dimension
        nb.typed.List.empty_list(nb.types.int64)
        for _ in range(cell_list_size)
	])
    for _ in range(move_count):
        # we want the cell list up-to-date before anything else
        bin_indices = np.floor(divisions * x / simulation_box_size) % divisions
        __repopulate_cell_list(
            bin_indices, cell_list, cell_list_size,
			particle_number, spatial_dimension_count,
			divisions
        )
        # executing the sampling moves where we update ...
		# ... non-interacting cells in parallel, i.e. one of ...
        # ... 3^d groups of cells which are all out of ...
        # ... interaction range of each other
        # TODO 	DO
        u(0, overlap_energy, sigma)
        # shuffling all particles to not accidentally introduce ...
        # ... effects only due to the fixed update order
        shuffle(x)


In [ ]:
run(
    1, # overlap energy
    1, # kBT
    0.3, # packing density
    1000, # particle number
    )